In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/india_flight_delay_model_input.csv")

features = [
    "Airline",
    "Origin_Airport",
    "Destination_Airport",
    "Scheduled_Departure_Hour",
    "Day_of_Week",
    "Month",
    "Is_Weekend",
    "Peak_Hour",
]
X = df[features]
y = df["Delay_Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


Train shape: (120000, 8) Test shape: (30000, 8)


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

cat_cols = ["Airline", "Origin_Airport", "Destination_Airport"]
num_cols = ["Scheduled_Departure_Hour", "Day_of_Week", "Month", "Is_Weekend", "Peak_Hour"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols),
])

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=15, class_weight="balanced", n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=6, scale_pos_weight=pos_weight, eval_metric="logloss", random_state=42),
}

for name, clf in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", clf)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    print(f"{name} -> Acc: {accuracy_score(y_test, preds):.3f} | F1: {f1_score(y_test, preds):.3f} | ROC-AUC: {roc_auc_score(y_test, probs):.3f}")

Logistic Regression -> Acc: 0.820 | F1: 0.888 | ROC-AUC: 0.835
Random Forest -> Acc: 0.797 | F1: 0.874 | ROC-AUC: 0.820
XGBoost -> Acc: 0.872 | F1: 0.924 | ROC-AUC: 0.837


In [3]:
import os
import joblib

os.makedirs("../backend/ml", exist_ok=True)

best_model = Pipeline([
    ("prep", preprocessor),
    ("model", models["XGBoost"])
])
best_model.fit(X_train, y_train)
joblib.dump(best_model, "../backend/ml/model.pkl")
print("Saved model to backend/ml/model.pkl!")

# Quick test prediction
sample = pd.DataFrame([{
    "Airline": "IndiGo",
    "Origin_Airport": "BOM",
    "Destination_Airport": "DEL",
    "Scheduled_Departure_Hour": 8,
    "Day_of_Week": 2,
    "Month": 12,
    "Is_Weekend": 0,
    "Peak_Hour": 1,
}])
prob = best_model.predict_proba(sample)[0][1]
print(f"Sample Delay Risk: {prob * 100:.1f}%")


Saved model to backend/ml/model.pkl!
Sample Delay Risk: 84.3%
